# S3_NB0 — does exit quality predict the oracle excess?

**~15 minutes · CPU only · no training · reads data already on disk**

## Why this runs first

Study 2's core claim is that the oracle early-exit bound sits **+6.86 points
above the network's own full-compute accuracy**, and that the excess is
per-exit noise. Study 1 trained its exit heads on a **frozen** backbone;
MSDNet, BranchyNet and DE3-BERT train exits **jointly**. Better exits might
shrink the excess to nothing, which would make Study 2 an artifact.

Settling that costs ~10 GPU-hours (`S3_NB1`). **This notebook tries to
pre-answer it for free.**

Across Study 1's 15 architectures, exit-head quality already varies a great
deal. If the excess **shrinks as the early exits get stronger**, joint training
would shrink it further and we can estimate by how much. If the excess is
**insensitive** to exit quality, joint training is unlikely to remove it.

> **This is a gate, not evidence.** It is observational and
> cross-architectural: architectures differ in many ways besides exit quality.
> It cannot replace `S3_NB1`. It only tells us how much `S3_NB1` is worth, and
> what to expect — which makes the GPU result falsifiable in advance instead of
> merely interesting.

In [ ]:
# === CELL 1 of every notebook: unpack the library ==========================
# This writes two Python files into the session and imports them. Nothing here
# touches the GPU or the network beyond installing three small packages.
#
#   msc_lib   f737f75f9303   the pipeline: HuggingFace sync, model zoo,
#                              measurement, training, the method
#   msc_core  2cc4ba5e0935   the reference maths: the MSC definition and
#                              every statistic in the paper
#
# Both are generated from KD/src by build_notebooks.py. Editing them HERE does
# nothing useful -- the next rebuild overwrites it. Edit the source instead.
import base64, os, subprocess, sys
from pathlib import Path

WORK = Path('/kaggle/working') if Path('/kaggle/working').is_dir() else Path.cwd()

# Kaggle images already ship torch, pandas and sklearn. These three vary by
# image version, so we check rather than assume.
#   pyarrow  writes the per-image measurement tables (Parquet)
#   pynvml   reads GPU power/temperature/utilisation directly
#   fvcore   counts FLOPs, which is how compute cost is defined
for _pkg in ('pyarrow', 'pynvml', 'fvcore', 'psutil'):
    try:
        __import__(_pkg)
    except ImportError:
        print(f'[BOOT] installing {_pkg} ...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _pkg,
                        '--break-system-packages'], check=False)

_LIB = (
    '...',
    'ciB0aGUgb2ZmbGluZSBjaGVja3MiKQo=',
)

_CORE = (
    '...',
)

for _name, _blob in (('msc_lib', _LIB), ('msc_core', _CORE)):
    (WORK / f'{_name}.py').write_bytes(base64.b64decode(''.join(_blob)))
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
for _m in [m for m in list(sys.modules) if m in ('msc_lib', 'msc_core')]:
    del sys.modules[_m]

import msc_lib as msc
import msc_core

print(f'[BOOT] msc_lib v{msc.__version__} ready  (torch available: {msc._TORCH_OK})')
print(f'[BOOT] artifact space: {msc.WORK_ROOT}   scratch space: {msc.SCRATCH_ROOT}')

In [ ]:
# === CELL 2 -- WHERE EVERYTHING LIVES ======================================
# Leave both as None and they are CHOSEN FOR YOU: the roomiest drive that
# actually exists on this machine. Set MSC_ROOT explicitly if your runs are
# somewhere specific -- e.g. r'C:\msc_results'.
# This notebook only READS existing runs -- it trains nothing, so
# DATA_DIR is irrelevant to it.

DATA_DIR = None      # e.g. r'E:\msc_data'      -- None = choose for me
MSC_ROOT = None      # e.g. r'C:\msc_results'   -- None = choose for me

# WHERE CIFAR-100 ALREADY IS. Point this at the folder that contains
# `cifar-100-python` (or at that folder itself -- both work). It is checked
# BEFORE any download path, so nothing ever re-fetches 169 MB over a copy you
# already have.
CIFAR_DIR = r'C:\Users\Administrator\Desktop\New folder'

# ---------------------------------------------------------------------------
import os
from pathlib import Path

M = msc                       # Study 2's notebooks say `M`; same module.

if CIFAR_DIR:
    os.environ['MSC_CIFAR_DIR'] = str(CIFAR_DIR)

# Study 3 is CIFAR-100. The library's default repo is the ImageNet one, so
# without this every push would land in msc-imagenet100 -- which is exactly what
# happened on the first joint run.
os.environ.setdefault('MSC_HF_REPO', 'Shanmuk4622/msc-cifar100')

_paths = M.resolve_storage(DATA_DIR, MSC_ROOT)
if not _paths['ok']:
    raise SystemExit('storage is not usable -- see the problems listed above')

DATA_DIR = _paths['data_dir']
MSC_ROOT = _paths['results_root']
os.environ['MSC_SCRATCH'] = MSC_ROOT

# OFFLINE BY DEFAULT. This machine does not always have a network, and a
# background uploader that retries mid-epoch turns a missing connection into a
# failed run. Everything is written to disk in full; S3_NB5_Publish uploads it
# in ONE pass at the end. `enable_hf` is the only switch that decides this.
sess = M.Session(account='local', phase='analysis', dataset='cifar100',
                 work_root=MSC_ROOT, session_limit_h=0.0,
                 enable_hf=False,
                 worker_id=0, num_workers=1)
print('HuggingFace: ' + ('ON -- publishing' if False else
                         'OFF -- fully offline, nothing is uploaded here'))

print(f'msc_lib   {M.__version__}')
print(f'MSC_ROOT  {MSC_ROOT}')
print(f'data_dir  {sess.data_dir}')

# Resolve CIFAR-100 now, loudly, rather than discovering mid-training that it
# is about to download. `_has_cifar100` is the same check the loader uses.
if CIFAR_DIR:
    _cd = Path(CIFAR_DIR)
    _cd = _cd if M._has_cifar100(_cd) else _cd.parent
    if M._has_cifar100(_cd):
        print(f'CIFAR-100  {_cd}  (found -- no download)')
    else:
        print(f'CIFAR-100  NOT at {CIFAR_DIR} -- expected a cifar-100-python/'
              ' folder with train/ and test/ inside. It will try to download.')

# A Session CREATES runs/, so "the directory exists" proves nothing. Count the
# runs that actually carry a measurement -- that is what every cell below reads.
_runs_dir = Path(MSC_ROOT) / 'runs'
_measured = sorted(d.name for d in _runs_dir.iterdir()
                   if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists()
                   ) if _runs_dir.is_dir() else []
print(f'measured runs on disk: {len(_measured)}')
if not _measured:
    raise RuntimeError(
        f'no measured runs under {MSC_ROOT}/runs.\n'
        'Study 3 re-analyses Study 1 output. Fetch it first with\n'
        'notebooks_study2/S2_NB0_Fetch.ipynb, or set MSC_ROOT above\n'
        'to the folder that already holds them.')
print(f'  e.g. {_measured[0]}')

In [ ]:
# === Which runs am I analysing? ===========================================
# One accessor, so the dedupe rule and the emptiness checks live in a single
# place rather than being re-typed in each notebook (rule 4).
import numpy as np, pandas as pd

runs_dir = Path(MSC_ROOT) / 'runs'

def measured_runs(dataset='cifar100', methods=('base',), require=True,
                  include_probes=False):
    '''Runs on disk that belong to the STUDY POPULATION.

    D-90. This walks runs/, and `atlas=False` governs what gets PLANNED, not
    what a directory scan finds. Once Study 4 trains
    `p7-msdnet-cifar100-jointexit-s1`, an unfiltered scan for jointexit runs
    would fold a DESIGNED-exit architecture into Study 3's attached-exit
    population and move numbers that are already written up -- silently,
    because a new run looks exactly like a run that was always there.
    '''
    ids = sorted(d.name for d in runs_dir.iterdir()
                 if d.is_dir() and (d / 'per_sample' / 'test.parquet').exists())
    if not ids:
        raise RuntimeError(f'no measured runs under {runs_dir}')
    df = pd.DataFrame([{**M.parse_run_id(r), 'run_id': r} for r in ids])

    for col in ('method', 'dataset', 'arch', 'seed', 'phase'):
        if col not in df.columns:
            raise RuntimeError(
                f'parse_run_id did not yield a {col!r} column. Columns present: '
                f'{list(df.columns)}. Run ids look like: {ids[:3]}')

    if not include_probes:
        probes = {a for a, m in M.ZOO.items() if not m.get('atlas', True)}
        hit = df[df['arch'].isin(probes)]
        df = df[~df['arch'].isin(probes)]
        if len(hit):
            print(f'excluded {len(hit)} probe run(s) from the study '
                  f'population: {sorted(hit["run_id"])}')

    sel = df[(df['dataset'] == dataset) & (df['method'].isin(methods))]
    if require and sel.empty:
        raise RuntimeError('; '.join([
            f'{len(df)} measured run(s) on disk, but NONE with '
            f'dataset={dataset!r} and method in {tuple(methods)!r}',
            f'datasets present: {sorted(df["dataset"].dropna().unique())}',
            f'methods present: {sorted(df["method"].dropna().unique())}',
            'either the wrong MSC_ROOT is set, or the runs you need have not '
            'been fetched or trained yet']))

    # p0 pilots and p1 runs share seed numbers, so pooling them counts one seed
    # twice -- the contamination Study 2 found. Keep the highest phase.
    before = len(sel)
    sel = (sel.sort_values('phase')
              .drop_duplicates(subset=['arch', 'dataset', 'method', 'seed'],
                               keep='last'))
    if len(sel) < before:
        print(f'dropped {before - len(sel)} duplicate (arch, method, seed) '
              f'run(s) -- pilot replicates')
    return sel.reset_index(drop=True)

_b = measured_runs()
print(f'{len(_b)} CIFAR-100 base run(s); '
      f'{_b["arch"].nunique()} architecture(s)')

---
## The quantities

For every measured run, from `per_sample/test.parquet`:

| | |
|---|---|
| `acc_full` | accuracy of the final exit — what you get by running everything |
| `acc_k` | accuracy of exit *k* alone |
| **`excess`** | `P(correct at ANY exit) − acc_full` — Study 2's +6.86 pt |
| **`exit_quality`** | mean of `acc_k / acc_full` over the early exits |

`exit_quality` near 1.0 means the early exits are nearly as good as the full
network. Frozen post-hoc heads should sit well below that; jointly trained ones
should sit closer to it. The regression of `excess` on `exit_quality` is the
extrapolation.

In [ ]:

import numpy as np, pandas as pd, itertools
from pathlib import Path
from scipy.stats import spearmanr

runs_dir = Path(MSC_ROOT) / 'runs'
base = measured_runs()          # THE accessor -- checks and dedupes

rows = []
for r in sorted(base['run_id']):
    d = pd.read_parquet(runs_dir / r / 'per_sample' / 'test.parquet')
    ks = sorted(int(c.split('_d')[1]) for c in d.columns
                if c.startswith('pred_d') and c.split('_d')[1].isdigit())
    lab = d['label'].to_numpy()
    corr = np.stack([(d[f'pred_d{k}'].to_numpy() == lab) for k in ks], axis=1)
    acc_k = corr.mean(axis=0)
    acc_full = float(acc_k[-1])
    rows.append({
        'run_id': r, 'arch': M.parse_run_id(r)['arch'],
        'seed': M.parse_run_id(r)['seed'],
        'acc_full': acc_full,
        'excess': float(corr.any(axis=1).mean()) - acc_full,
        'exit_quality': float(np.mean(acc_k[:-1] / max(acc_full, 1e-9))),
        'acc_first_exit': float(acc_k[0]),
        'K': len(ks),
    })

ex = pd.DataFrame(rows)
M.save_analysis(sess.data_dir, 's3_exit_quality', ex)
per = ex.groupby('arch')[['acc_full', 'excess', 'exit_quality',
                          'acc_first_exit']].mean()
print(per.round(4).sort_values('exit_quality').to_string())

---
## The extrapolation

`excess` regressed on `exit_quality`. The **slope** is what matters: it says how
many accuracy points the excess falls for each unit of improvement in the early
exits.

We do not know where jointly trained exits land on the `exit_quality` axis, so
the cell reports the predicted excess at several plausible values instead of
guessing one. `S3_NB1` will measure the real value and we compare.

In [ ]:

r_q, p_q = spearmanr(ex['exit_quality'], ex['excess'])
print(f'Spearman(exit_quality, excess) = {r_q:+.3f}   p = {p_q:.4f}   '
      f'n = {len(ex)} runs')

# THE CONFOUND, and it is not subtle: exit_quality = mean(acc_k / acc_full), so
# acc_full is its own denominator. A low-accuracy network scores high
# exit_quality with weak early exits -- and a low-accuracy network makes more
# final-layer errors, which is exactly what `excess` counts. Reporting the raw
# correlation alone would be reporting that circularity as a finding.
r_a, p_a = spearmanr(ex['acc_full'], ex['excess'])
print(f'Spearman(acc_full,     excess) = {r_a:+.3f}   p = {p_a:.4f}   '
      '<- the confound')

def _partial(x, y, z):
    rx, ry, rz = (pd.Series(v).rank().to_numpy() for v in (x, y, z))
    res = lambda t: t - np.polyval(np.polyfit(rz, t, 1), rz)
    return spearmanr(res(rx), res(ry))

r_p, p_p = _partial(ex['exit_quality'], ex['excess'], ex['acc_full'])
print(f'PARTIAL, holding acc_full fixed = {r_p:+.3f}   p = {p_p:.4f}   '
      '<- the one that counts')
if abs(r_p) < 0.2:
    print('  -> the raw relationship was mostly the confound. Treat the')
    print('     extrapolation below as having no predictive power.')
print()

A = np.polyfit(ex['exit_quality'], ex['excess'], 1)
slope, intercept = float(A[0]), float(A[1])
print(f'OLS: excess = {slope:+.4f} * exit_quality {intercept:+.4f}')
print()

obs_lo, obs_hi = ex['exit_quality'].min(), ex['exit_quality'].max()
print(f'observed exit_quality range (frozen heads): '
      f'{obs_lo:.3f} to {obs_hi:.3f}')
print()
print('predicted excess if joint training reaches:')
for q in sorted({round(float(obs_hi), 4), 0.90, 0.95, 1.00}):
    pred = slope * q + intercept
    tag = '  <- best frozen run' if abs(q - obs_hi) < 1e-9 else ''
    extrap = '  (EXTRAPOLATION, outside observed range)' if q > obs_hi else ''
    print(f'    exit_quality {q:.2f}  ->  excess {pred*100:+6.2f} pt{tag}{extrap}')

print()
print('--- GATE ---')
pred_at_1 = slope * 1.0 + intercept
if p_q > 0.05:
    print('exit quality does NOT predict the excess (p > 0.05).')
    print('=> Joint training is unlikely to remove it. S3_NB1 is still needed')
    print('   to confirm, but expect the excess to SURVIVE.')
elif pred_at_1 * 100 > 2.0:
    print(f'even at exit_quality = 1.0 the predicted excess is '
          f'{pred_at_1*100:+.2f} pt, above the 2.0 pt threshold.')
    print('=> Expect Study 2 to SURVIVE joint training. Run S3_NB1 to confirm.')
else:
    print(f'at exit_quality = 1.0 the predicted excess is '
          f'{pred_at_1*100:+.2f} pt, below the 2.0 pt threshold.')
    print('=> Study 2 may be a frozen-backbone ARTIFACT. S3_NB1 is now the')
    print('   most important experiment in the project -- run it next.')
print()
print('Either way this is observational. It sets an EXPECTATION that S3_NB1')
print('can falsify, which is the point of running it first.')

---
## Canary — this analysis must be able to give a wrong answer

Three synthetic checks. The load-bearing one is the third: if `excess` were
computed wrongly it might read ~0 everywhere, and "no excess" and "cannot
measure excess" look identical in the output.

In [ ]:

def _excess_of(corr):
    a = corr.mean(axis=0)
    return float(corr.any(axis=1).mean()) - float(a[-1])

rng = np.random.default_rng(0)
n, K = 5000, 5

# 1. perfect exits, all identical -> excess must be EXACTLY zero
c = np.zeros((n, K)); c[rng.random(n) < 0.7, :] = 1
e1 = _excess_of(c)
print(f'{"PASS" if abs(e1) < 1e-12 else "FAIL"}  identical exits -> excess '
      f'{e1*100:+.4f} pt (must be 0)')

# 2. early exits right where the final is wrong -> excess must be large
c = np.zeros((n, K)); hit = rng.random(n) < 0.3
c[hit, 0] = 1                      # right early, wrong at the end
c[rng.random(n) < 0.6, -1] = 1
e2 = _excess_of(c)
print(f'{"PASS" if e2 > 0.05 else "FAIL"}  early-only correctness -> excess '
      f'{e2*100:+.2f} pt (must be large)')

# 3. excess can never be negative, on any input
worst = min(_excess_of((rng.random((400, K)) < rng.random()).astype(float))
            for _ in range(300))
print(f'{"PASS" if worst >= -1e-12 else "FAIL"}  excess is never negative '
      f'over 300 random draws (worst {worst*100:+.6f} pt)')

---
## What to do next

Read the **GATE** block above, then:

| gate says | next |
|---|---|
| excess survives at `exit_quality = 1.0` | run **`S3_NB1`** to confirm — it should agree |
| excess vanishes | run **`S3_NB1`** anyway; it is now the highest-value experiment in the project, because it decides whether Study 2 stands |
| exit quality does not predict excess | run **`S3_NB1`**; the extrapolation has no power and only the real experiment will settle it |

**In every branch the answer is `S3_NB1`.** What changes is what you should
expect — and having written the expectation down first is what makes the GPU
result meaningful rather than merely a number.

Record the outcome in `study3/03_LOG.md` before moving on.